In [26]:
import pandas as pd
import numpy as np

keyboard_features = pd.read_csv(
    "../data/processed/keyboard_features.csv"
)

print("Shape:", keyboard_features.shape)

Shape: (10000, 33)


In [27]:
print("Missing values:", keyboard_features.isna().sum().sum())

print(
    "Duplicate window IDs:",
    keyboard_features.duplicated(
        subset=["participant", "session", "window_id"]
    ).sum()
)

Missing values: 0
Duplicate window IDs: 0


In [28]:
feature_cols = [
    col for col in keyboard_features.columns
    if col not in ["participant", "session", "window_id"]
]

print("Number of feature columns:", len(feature_cols))
print(feature_cols)

Number of feature columns: 30
['DU.key1.key1_mean', 'DU.key1.key1_std', 'DU.key1.key1_median', 'DU.key1.key1_min', 'DU.key1.key1_max', 'DD.key1.key2_mean', 'DD.key1.key2_std', 'DD.key1.key2_median', 'DD.key1.key2_min', 'DD.key1.key2_max', 'DU.key1.key2_mean', 'DU.key1.key2_std', 'DU.key1.key2_median', 'DU.key1.key2_min', 'DU.key1.key2_max', 'UD.key1.key2_mean', 'UD.key1.key2_std', 'UD.key1.key2_median', 'UD.key1.key2_min', 'UD.key1.key2_max', 'UU.key1.key2_mean', 'UU.key1.key2_std', 'UU.key1.key2_median', 'UU.key1.key2_min', 'UU.key1.key2_max', 'unique_key1_count', 'unique_key2_count', 'unique_transition_count', 'consecutive_key1_repetition_rate', 'backspace_rate']


In [29]:
feature_stats = keyboard_features[feature_cols].describe().T

print(feature_stats)

                                    count       mean        std         min  \
DU.key1.key1_mean                 10000.0   0.202839   0.642578    0.000000   
DU.key1.key1_std                  10000.0   0.637142   2.800346    0.000000   
DU.key1.key1_median               10000.0   0.093991   0.022237    0.000000   
DU.key1.key1_min                  10000.0   0.043946   0.019184    0.000000   
DU.key1.key1_max                  10000.0   4.290890  17.861060    0.000000   
DD.key1.key2_mean                 10000.0   0.296444   0.706292   -2.757180   
DD.key1.key2_std                  10000.0   1.333522   5.994021    0.005731   
DD.key1.key2_median               10000.0   0.179081   0.050174    0.026000   
DD.key1.key2_min                  10000.0  -4.000562  17.741652 -568.083000   
DD.key1.key2_max                  10000.0   7.063721  37.836070    0.045000   
DU.key1.key2_mean                 10000.0   0.499250   0.956893    0.032740   
DU.key1.key2_std                  10000.0   1.090069

In [30]:
percentile_stats = keyboard_features[feature_cols].quantile(
    [0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

print(percentile_stats)

                                       0.05       0.25       0.50       0.75  \
DU.key1.key1_mean                  0.068960   0.083840   0.098900   0.123425   
DU.key1.key1_std                   0.014871   0.023607   0.032482   0.051291   
DU.key1.key1_median                0.064000   0.079000   0.090500   0.107500   
DU.key1.key1_min                   0.005000   0.032000   0.046000   0.056000   
DU.key1.key1_max                   0.112000   0.160000   0.214000   0.358000   
DD.key1.key2_mean                  0.135235   0.188395   0.241720   0.312130   
DD.key1.key2_std                   0.081762   0.146087   0.226202   0.375513   
DD.key1.key2_median                0.112000   0.145000   0.173500   0.205500   
DD.key1.key2_min                 -21.333200  -0.162000   0.028000   0.065000   
DD.key1.key2_max                   0.432000   0.746000   1.120000   1.985250   
DU.key1.key2_mean                  0.230790   0.298610   0.360700   0.450990   
DU.key1.key2_std                   0.081

In [31]:
robust_ranges = keyboard_features[feature_cols].quantile(
    [0.05, 0.95]
).T

robust_ranges.columns = ["p05", "p95"]

print(robust_ranges)

                                        p05        p95
DU.key1.key1_mean                  0.068960   0.698088
DU.key1.key1_std                   0.014871   3.362709
DU.key1.key1_median                0.064000   0.134000
DU.key1.key1_min                   0.005000   0.074000
DU.key1.key1_max                   0.112000  22.176950
DD.key1.key2_mean                  0.135235   0.460001
DD.key1.key2_std                   0.081762   5.693080
DD.key1.key2_median                0.112000   0.265500
DD.key1.key2_min                 -21.333200   0.105000
DD.key1.key2_max                   0.432000  28.933100
DU.key1.key2_mean                  0.230790   1.108545
DU.key1.key2_std                   0.081304   4.264983
DU.key1.key2_median                0.206500   0.373000
DU.key1.key2_min                   0.038000   0.203000
DU.key1.key2_max                   0.527000  29.087050
UD.key1.key2_mean                 -0.433742   0.352070
UD.key1.key2_std                   0.084917   4.187720
UD.key1.ke

In [32]:
candidate_features = [
    "DU.key1.key1_median",
    "DD.key1.key2_median",
    "DU.key1.key2_median",
    "UD.key1.key2_median",
    "UU.key1.key2_median",
    "unique_key1_count",
    "unique_key2_count",
    "unique_transition_count",
    "consecutive_key1_repetition_rate",
    "backspace_rate"
]

correlation_matrix = keyboard_features[candidate_features].corr()

print(correlation_matrix.round(3))

                                  DU.key1.key1_median  DD.key1.key2_median  \
DU.key1.key1_median                             1.000               -0.014   
DD.key1.key2_median                            -0.014                1.000   
DU.key1.key2_median                             0.407                0.881   
UD.key1.key2_median                            -0.390                0.904   
UU.key1.key2_median                             0.031                0.975   
unique_key1_count                               0.151                0.175   
unique_key2_count                               0.150                0.176   
unique_transition_count                         0.222                0.189   
consecutive_key1_repetition_rate               -0.273               -0.230   
backspace_rate                                 -0.191               -0.176   

                                  DU.key1.key2_median  UD.key1.key2_median  \
DU.key1.key1_median                             0.407          

In [33]:
corr_pairs = (
    correlation_matrix
    .where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))
    .stack()
    .sort_values(key=abs, ascending=False)
)

print(corr_pairs)

unique_key1_count    unique_key2_count                   0.977206
DD.key1.key2_median  UU.key1.key2_median                 0.974973
                     UD.key1.key2_median                 0.904155
DU.key1.key2_median  UU.key1.key2_median                 0.898778
UD.key1.key2_median  UU.key1.key2_median                 0.885697
                                                           ...   
backspace_rate       unique_key1_count                        NaN
                     unique_key2_count                        NaN
                     unique_transition_count                  NaN
                     consecutive_key1_repetition_rate         NaN
                     backspace_rate                           NaN
Length: 100, dtype: float64


In [34]:
corr_with_behavior = correlation_matrix[
    [
        "unique_key1_count",
        "unique_key2_count",
        "unique_transition_count",
        "consecutive_key1_repetition_rate",
        "backspace_rate"
    ]
]

print(corr_with_behavior.round(3))

                                  unique_key1_count  unique_key2_count  \
DU.key1.key1_median                           0.151              0.150   
DD.key1.key2_median                           0.175              0.176   
DU.key1.key2_median                           0.220              0.221   
UD.key1.key2_median                           0.090              0.092   
UU.key1.key2_median                           0.178              0.179   
unique_key1_count                             1.000              0.977   
unique_key2_count                             0.977              1.000   
unique_transition_count                       0.671              0.674   
consecutive_key1_repetition_rate             -0.525             -0.522   
backspace_rate                               -0.459             -0.458   

                                  unique_transition_count  \
DU.key1.key1_median                                 0.222   
DD.key1.key2_median                                 0.189   
DU

In [35]:
selected_features = [
    "DU.key1.key1_median",
    "DU.key1.key2_median",
    "unique_key1_count",
    "unique_transition_count",
    "consecutive_key1_repetition_rate",
    "backspace_rate"
]

print("Selected features:")
for feature in selected_features:
    print("-", feature)

Selected features:
- DU.key1.key1_median
- DU.key1.key2_median
- unique_key1_count
- unique_transition_count
- consecutive_key1_repetition_rate
- backspace_rate


In [36]:
selected_corr = keyboard_features[selected_features].corr()

print(selected_corr.round(3))

                                  DU.key1.key1_median  DU.key1.key2_median  \
DU.key1.key1_median                             1.000                0.407   
DU.key1.key2_median                             0.407                1.000   
unique_key1_count                               0.151                0.220   
unique_transition_count                         0.222                0.252   
consecutive_key1_repetition_rate               -0.273               -0.304   
backspace_rate                                 -0.191               -0.234   

                                  unique_key1_count  unique_transition_count  \
DU.key1.key1_median                           0.151                    0.222   
DU.key1.key2_median                           0.220                    0.252   
unique_key1_count                             1.000                    0.671   
unique_transition_count                       0.671                    1.000   
consecutive_key1_repetition_rate             -0.525  

In [37]:
for feature in selected_features:
    print("\n", feature)
    print("Mean   :", keyboard_features[feature].mean())
    print("Median :", keyboard_features[feature].median())
    print("P05    :", keyboard_features[feature].quantile(0.05))
    print("P95    :", keyboard_features[feature].quantile(0.95))


 DU.key1.key1_median
Mean   : 0.09399054999999999
Median : 0.0905
P05    : 0.064
P95    : 0.134

 DU.key1.key2_median
Mean   : 0.27826175000000003
Median : 0.272
P05    : 0.2065
P95    : 0.373

 unique_key1_count
Mean   : 19.4746
Median : 20.0
P05    : 16.0
P95    : 23.0

 unique_transition_count
Mean   : 41.1919
Median : 42.0
P05    : 34.0
P95    : 47.0

 consecutive_key1_repetition_rate
Mean   : 0.05285102040816323
Median : 0.0408163265306122
P05    : 0.0
P95    : 0.1428571428571428

 backspace_rate
Mean   : 0.066746
Median : 0.04
P05    : 0.0
P95    : 0.18


In [38]:
percentile_table = pd.DataFrame({
    "P05": keyboard_features[selected_features].quantile(0.05),
    "Median": keyboard_features[selected_features].quantile(0.50),
    "P95": keyboard_features[selected_features].quantile(0.95)
})

print(percentile_table.round(4).to_string())

                                      P05   Median      P95
DU.key1.key1_median                0.0640   0.0905   0.1340
DU.key1.key2_median                0.2065   0.2720   0.3730
unique_key1_count                 16.0000  20.0000  23.0000
unique_transition_count           34.0000  42.0000  47.0000
consecutive_key1_repetition_rate   0.0000   0.0408   0.1429
backspace_rate                     0.0000   0.0400   0.1800


In [39]:
# Robust 5th–95th percentile normalization
def robust_normalize(series):
    p05 = series.quantile(0.05)
    p95 = series.quantile(0.95)
    
    normalized = (series - p05) / (p95 - p05)
    return normalized.clip(0, 1)


bpi_features = pd.DataFrame(index=keyboard_features.index)

# Timing features
bpi_features["timing_key1_norm"] = robust_normalize(
    keyboard_features["DU.key1.key1_median"]
)

bpi_features["timing_transition_norm"] = robust_normalize(
    keyboard_features["DU.key1.key2_median"]
)

# Activity / diversity
bpi_features["unique_key_norm"] = robust_normalize(
    keyboard_features["unique_key1_count"]
)

bpi_features["transition_diversity_norm"] = robust_normalize(
    keyboard_features["unique_transition_count"]
)

# Repetition and correction
bpi_features["repetition_norm"] = robust_normalize(
    keyboard_features["consecutive_key1_repetition_rate"]
)

bpi_features["backspace_norm"] = robust_normalize(
    keyboard_features["backspace_rate"]
)

print(bpi_features.describe().round(3))

       timing_key1_norm  timing_transition_norm  unique_key_norm  \
count         10000.000               10000.000        10000.000   
mean              0.427                   0.429            0.502   
std               0.285                   0.279            0.280   
min               0.000                   0.000            0.000   
25%               0.214                   0.210            0.286   
50%               0.379                   0.393            0.571   
75%               0.621                   0.619            0.714   
max               1.000                   1.000            1.000   

       transition_diversity_norm  repetition_norm  backspace_norm  
count                  10000.000        10000.000       10000.000  
mean                       0.572            0.314           0.336  
std                        0.261            0.287           0.304  
min                        0.000            0.000           0.000  
25%                        0.385            0.1

In [40]:
bpi_directional = pd.DataFrame(index=keyboard_features.index)

# Timing: lower timing -> higher efficiency
bpi_directional["timing_key1_score"] = 1 - bpi_features["timing_key1_norm"]
bpi_directional["timing_transition_score"] = 1 - bpi_features["timing_transition_norm"]

# Activity / diversity: higher -> higher score
bpi_directional["unique_key_score"] = bpi_features["unique_key_norm"]
bpi_directional["transition_diversity_score"] = bpi_features["transition_diversity_norm"]

# Repetition / correction: lower -> higher score
bpi_directional["repetition_score"] = 1 - bpi_features["repetition_norm"]
bpi_directional["backspace_score"] = 1 - bpi_features["backspace_norm"]

print(bpi_directional.describe().round(3))

       timing_key1_score  timing_transition_score  unique_key_score  \
count          10000.000                10000.000         10000.000   
mean               0.573                    0.571             0.502   
std                0.285                    0.279             0.280   
min                0.000                    0.000             0.000   
25%                0.379                    0.381             0.286   
50%                0.621                    0.607             0.571   
75%                0.786                    0.790             0.714   
max                1.000                    1.000             1.000   

       transition_diversity_score  repetition_score  backspace_score  
count                   10000.000         10000.000        10000.000  
mean                        0.572             0.686            0.664  
std                         0.261             0.287            0.304  
min                         0.000             0.000            0.000  
25%  

In [41]:
bpi_components = pd.DataFrame(index=keyboard_features.index)

# 1. Timing efficiency
bpi_components["timing_efficiency"] = (
    0.5 * bpi_directional["timing_key1_score"]
    + 0.5 * bpi_directional["timing_transition_score"]
)

# 2. Activity / diversity
bpi_components["activity_diversity"] = (
    0.5 * bpi_directional["unique_key_score"]
    + 0.5 * bpi_directional["transition_diversity_score"]
)

# 3. Typing behavior
bpi_components["typing_behavior"] = (
    0.5 * bpi_directional["repetition_score"]
    + 0.5 * bpi_directional["backspace_score"]
)

print(bpi_components.describe().round(3))

       timing_efficiency  activity_diversity  typing_behavior
count          10000.000           10000.000        10000.000
mean               0.572               0.537            0.675
std                0.234               0.238            0.276
min                0.000               0.000            0.000
25%                0.404               0.374            0.508
50%                0.596               0.555            0.746
75%                0.761               0.709            0.889
max                1.000               1.000            1.000


In [42]:
component_corr = bpi_components.corr()

print(component_corr.round(3))

                    timing_efficiency  activity_diversity  typing_behavior
timing_efficiency               1.000              -0.117           -0.100
activity_diversity             -0.117               1.000            0.336
typing_behavior                -0.100               0.336            1.000


In [43]:
# Final keyboard productivity score
keyboard_features["productivity_score"] = (
    100
    * (
        bpi_components["timing_efficiency"]
        + bpi_components["activity_diversity"]
        + bpi_components["typing_behavior"]
    )
    / 3
)

print(
    keyboard_features["productivity_score"]
    .describe()
    .round(3)
)

count    10000.000
mean        59.460
std         15.113
min          3.690
25%         49.954
50%         61.058
75%         70.625
max         97.436
Name: productivity_score, dtype: float64


In [44]:
print("Missing scores:", keyboard_features["productivity_score"].isna().sum())
print("Scores below 0:", (keyboard_features["productivity_score"] < 0).sum())
print("Scores above 100:", (keyboard_features["productivity_score"] > 100).sum())
print("Number of scores:", len(keyboard_features))

print("\nSample:")
print(
    pd.concat(
        [
            bpi_components,
            keyboard_features["productivity_score"]
        ],
        axis=1
    ).head(10).round(3)
)

Missing scores: 0
Scores below 0: 0
Scores above 100: 0
Number of scores: 10000

Sample:
   timing_efficiency  activity_diversity  typing_behavior  productivity_score
0              0.484               0.440            1.000              64.128
1              0.458               0.412            0.929              59.952
2              0.383               0.484            0.929              59.828
3              0.638               0.484            0.563              56.156
4              0.462               0.813            0.857              71.067
5              0.525               0.231            0.786              51.382
6              0.497               0.368            0.929              59.787
7              0.657               0.412            1.000              68.971
8              0.568               0.742            1.000              76.993
9              0.402               0.484            1.000              62.838


In [45]:
scored_keyboard_features = keyboard_features.copy()

output_path = "../data/processed/keyboard_productivity_scores.csv"

scored_keyboard_features.to_csv(
    output_path,
    index=False
)

print("Saved successfully:", output_path)
print("Shape:", scored_keyboard_features.shape)

Saved successfully: ../data/processed/keyboard_productivity_scores.csv
Shape: (10000, 34)


In [46]:
import pandas as pd
import numpy as np

scored_keyboard_features = pd.read_csv(
    "../data/processed/keyboard_productivity_scores.csv"
)

print("Shape:", scored_keyboard_features.shape)
print(scored_keyboard_features.head())

Shape: (10000, 34)
  participant  session  window_id  DU.key1.key1_mean  DU.key1.key1_std  \
0        p001        1          0            0.12090          0.126031   
1        p001        1          1            0.10006          0.035225   
2        p001        1          2            0.10054          0.031475   
3        p001        1          3            0.10018          0.041348   
4        p001        1          4            0.15258          0.248436   

   DU.key1.key1_median  DU.key1.key1_min  DU.key1.key1_max  DD.key1.key2_mean  \
0               0.0950             0.056             0.962            0.31160   
1               0.0865             0.055             0.184            0.69992   
2               0.0945             0.059             0.164            0.36300   
3               0.0895             0.051             0.304            0.28254   
4               0.1030             0.054             1.595            0.65990   

   DD.key1.key2_std  ...  UU.key1.key2_std  UU.ke

In [47]:
feature_cols = [
    col for col in scored_keyboard_features.columns
    if col not in [
        "participant",
        "session",
        "window_id",
        "productivity_score"
    ]
]

X = scored_keyboard_features[feature_cols]

y = scored_keyboard_features["productivity_score"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of features:", len(feature_cols))

X shape: (10000, 30)
y shape: (10000,)
Number of features: 30


In [48]:
# Create a unique group for each participant-session
groups = (
    scored_keyboard_features["participant"].astype(str)
    + "_session_"
    + scored_keyboard_features["session"].astype(str)
)

print("Total windows:", len(scored_keyboard_features))
print("Unique participant-session groups:", groups.nunique())
print("\nWindows per group:")
print(groups.value_counts().describe())

Total windows: 10000
Unique participant-session groups: 197

Windows per group:
count    197.000000
mean      50.761421
std        4.391426
min       43.000000
25%       48.000000
50%       50.000000
75%       52.000000
max       83.000000
Name: count, dtype: float64


In [49]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [50]:
from sklearn.model_selection import GroupShuffleSplit

print("scikit-learn is working!")

scikit-learn is working!


In [51]:
from sklearn.model_selection import GroupShuffleSplit

# 80% groups for training, 20% groups for testing
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining groups:", groups_train.nunique())
print("Testing groups:", groups_test.nunique())

Training samples: 7971
Testing samples: 2029

Training groups: 157
Testing groups: 40


In [52]:
overlap = set(groups_train).intersection(set(groups_test))

print("Overlapping participant-session groups:", len(overlap))
print("Overlap:", overlap)

Overlapping participant-session groups: 0
Overlap: set()


In [53]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

X_train_scaled shape: (7971, 30)
X_test_scaled shape: (2029, 30)


In [54]:
scaler.fit_transform(X_train)

array([[-0.14287661, -0.18937867,  0.09509711, ..., -0.67175353,
        -0.6123008 , -0.75444721],
       [-0.17889916, -0.22227701, -0.29980127, ..., -0.03359835,
        -0.37908933, -0.75444721],
       [-0.17806947, -0.22363541,  0.07186779, ..., -0.03359835,
        -0.37908933, -0.75444721],
       ...,
       [-0.17340244, -0.2242788 ,  0.14155574, ...,  0.39183843,
        -0.37908933, -0.53134579],
       [ 1.10785272,  1.68865791, -0.29980127, ...,  0.17912004,
         0.08733361,  0.36105988],
       [-0.19476706, -0.22302939, -0.25334264, ..., -1.5226271 ,
         1.02017949,  1.47656696]], shape=(7971, 30))

In [55]:
scaler.transform(X_test)

array([[-0.22515461, -0.21879453, -0.60178239, ..., -1.73534549,
         0.78696802,  0.80726271],
       [-0.22498176, -0.22565816, -0.99668077, ...,  0.81727522,
         0.08733361,  0.13795846],
       [-0.21260549, -0.22661912, -0.55532376, ..., -0.67175353,
        -0.37908933, -0.53134579],
       ...,
       [-0.20351342, -0.2167259 , -0.69469966, ...,  0.17912004,
        -0.37908933, -0.75444721],
       [-0.20178489, -0.22131396, -0.02104947, ..., -0.45903514,
         0.55375655, -0.53134579],
       [-0.23338241, -0.21754836, -0.95022214, ..., -0.88447192,
         2.18623684, -0.53134579]], shape=(2029, 30))

In [56]:
scaler.fit_transform(X_test)

array([[-0.14934026, -0.18521139, -0.74004381, ..., -2.13793878,
         1.19998012,  1.20895182],
       [-0.1492224 , -0.19162978, -1.08822669, ...,  0.94366988,
         0.18620094,  0.2636866 ],
       [-0.14078402, -0.1925284 , -0.69908112, ..., -0.85393517,
        -0.48965185, -0.68157862],
       ...,
       [-0.13458487, -0.18327695, -0.82196919, ...,  0.17326771,
        -0.48965185, -0.99666703],
       [-0.13340632, -0.18756738, -0.22801017, ..., -0.59713445,
         0.86205372, -0.68157862],
       [-0.15495013, -0.18404606, -1.047264  , ..., -1.11073589,
         3.22753847, -0.68157862]], shape=(2029, 30))

In [57]:
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

print("\nTraining mean (first 5 features):")
print(X_train_scaled.mean(axis=0)[:5])

print("\nTraining std (first 5 features):")
print(X_train_scaled.std(axis=0)[:5])

X_train_scaled shape: (7971, 30)
X_test_scaled shape: (2029, 30)

Training mean (first 5 features):
[ 0.00000000e+00  0.00000000e+00  1.78281956e-16 -1.06969174e-16
 -2.85251130e-17]

Training std (first 5 features):
[1. 1. 1. 1. 1.]


In [58]:
from sklearn.linear_model import LinearRegression

# Create the Linear Regression model
linear_model = LinearRegression()

# Train only on the training data
linear_model.fit(X_train_scaled, y_train)

print("Linear Regression model trained successfully.")

Linear Regression model trained successfully.


In [59]:
y_pred = linear_model.predict(X_test_scaled)

print("Number of predictions:", len(y_pred))
print("First 10 predictions:")
print(y_pred[:10])

Number of predictions: 2029
First 10 predictions:
[45.38190804 64.87751123 66.80992361 65.13961908 71.73921678 84.35405103
 53.69835578 69.36005135 67.21364243 74.35486883]


In [60]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Linear Regression Results")
print("-------------------------")
print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R²  :", r2)

Linear Regression Results
-------------------------
MAE : 5.267042038197612
MSE : 77.2036431529618
RMSE: 8.786560370984871
R²  : 0.6514277896533445


In [61]:
coefficients = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": linear_model.coef_
})

coefficients["absolute_coefficient"] = coefficients["coefficient"].abs()

coefficients = coefficients.sort_values(
    "absolute_coefficient",
    ascending=False
)

print(coefficients.to_string(index=False))

                         feature  coefficient  absolute_coefficient
                UU.key1.key2_std    22.140389             22.140389
                UD.key1.key2_min    21.993932             21.993932
                DU.key1.key2_std   -21.405367             21.405367
                DD.key1.key2_min   -18.009241             18.009241
                DU.key1.key2_max    13.692740             13.692740
                DD.key1.key2_max   -11.704594             11.704594
                DD.key1.key2_std    10.588792             10.588792
                UD.key1.key2_max    -8.918298              8.918298
             DU.key1.key1_median    -7.778456              7.778456
               UU.key1.key2_mean    -7.198422              7.198422
                  backspace_rate    -5.740770              5.740770
               unique_key1_count     5.034045              5.034045
                DU.key1.key1_max     4.832975              4.832975
                UD.key1.key2_std     3.995780   

In [62]:
print("Top 10 Positive Coefficients")
print("--------------------------------")

positive_coefficients = coefficients.sort_values(
    "coefficient",
    ascending=False
).head(10)

print(
    positive_coefficients[
        ["feature", "coefficient"]
    ].to_string(index=False)
)


print("\nTop 10 Negative Coefficients")
print("--------------------------------")

negative_coefficients = coefficients.sort_values(
    "coefficient",
    ascending=True
).head(10)

print(
    negative_coefficients[
        ["feature", "coefficient"]
    ].to_string(index=False)
)

Top 10 Positive Coefficients
--------------------------------
                feature  coefficient
       UU.key1.key2_std    22.140389
       UD.key1.key2_min    21.993932
       DU.key1.key2_max    13.692740
       DD.key1.key2_std    10.588792
      unique_key1_count     5.034045
       DU.key1.key1_max     4.832975
       UD.key1.key2_std     3.995780
unique_transition_count     3.833943
      DU.key1.key1_mean     2.841683
      DD.key1.key2_mean     2.349269

Top 10 Negative Coefficients
--------------------------------
            feature  coefficient
   DU.key1.key2_std   -21.405367
   DD.key1.key2_min   -18.009241
   DD.key1.key2_max   -11.704594
   UD.key1.key2_max    -8.918298
DU.key1.key1_median    -7.778456
  UU.key1.key2_mean    -7.198422
     backspace_rate    -5.740770
  DU.key1.key2_mean    -3.852064
DU.key1.key2_median    -3.129371
UU.key1.key2_median    -0.867404


In [63]:
corr_matrix = X.corr().abs()

# Keep only the upper triangle so each pair appears once
upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_pairs = (
    upper_triangle
    .stack()
    .sort_values(ascending=False)
)

high_corr_pairs = high_corr_pairs[
    high_corr_pairs >= 0.80
]

print("Highly correlated feature pairs (|r| >= 0.80):")
print("-----------------------------------------------")
print(high_corr_pairs.to_string())

Highly correlated feature pairs (|r| >= 0.80):
-----------------------------------------------
UD.key1.key2_max                  UU.key1.key2_max       0.999945
DD.key1.key2_min                  UD.key1.key2_min       0.999908
UU.key1.key2_std                  UU.key1.key2_max       0.999890
UD.key1.key2_max                  UU.key1.key2_std       0.999832
DU.key1.key2_std                  UD.key1.key2_std       0.997066
DD.key1.key2_mean                 UU.key1.key2_mean      0.995631
DU.key1.key2_std                  DU.key1.key2_max       0.995146
DD.key1.key2_max                  DU.key1.key2_max       0.994301
UU.key1.key2_mean                 UU.key1.key2_std       0.993402
                                  UU.key1.key2_max       0.992344
UD.key1.key2_max                  UU.key1.key2_mean      0.992278
DU.key1.key2_max                  UD.key1.key2_std       0.992113
DD.key1.key2_mean                 UU.key1.key2_std       0.989051
                                  UU.key1.key2_

In [64]:
thresholds = [0.80, 0.90, 0.95, 0.99]

print("Number of highly correlated feature pairs")
print("-----------------------------------------")

for threshold in thresholds:
    count = (high_corr_pairs >= threshold).sum()
    print(f"|r| >= {threshold:.2f}: {count} pairs")

Number of highly correlated feature pairs
-----------------------------------------
|r| >= 0.80: 67 pairs
|r| >= 0.90: 37 pairs
|r| >= 0.95: 28 pairs
|r| >= 0.99: 12 pairs


In [65]:
target_features = [
    "DU.key1.key1_median",
    "DU.key1.key2_median",
    "unique_key1_count",
    "unique_transition_count",
    "consecutive_key1_repetition_rate",
    "backspace_rate"
]

target_feature_corr = X.corr()[target_features].abs()

print("Correlation of all features with the six target-construction features:")
print("-----------------------------------------------------------------------")

print(
    target_feature_corr
    .round(3)
    .sort_values(
        by=target_features,
        ascending=False
    )
)

Correlation of all features with the six target-construction features:
-----------------------------------------------------------------------
                                  DU.key1.key1_median  DU.key1.key2_median  \
DU.key1.key1_median                             1.000                0.407   
DU.key1.key1_min                                0.477                0.197   
DU.key1.key2_median                             0.407                1.000   
UD.key1.key2_median                             0.390                0.634   
consecutive_key1_repetition_rate                0.273                0.304   
unique_transition_count                         0.222                0.252   
backspace_rate                                  0.191                0.234   
unique_key1_count                               0.151                0.220   
unique_key2_count                               0.150                0.221   
DU.key1.key2_min                                0.132                0.146   

In [66]:
print("Strongest correlation with each target-construction feature")
print("------------------------------------------------------------")

for feature in target_features:
    correlations = X.corr()[feature].drop(feature).abs()
    
    strongest_feature = correlations.idxmax()
    strongest_value = correlations.max()
    
    print(
        f"{feature:<40} "
        f"{strongest_feature:<35} "
        f"{strongest_value:.3f}"
    )

Strongest correlation with each target-construction feature
------------------------------------------------------------
DU.key1.key1_median                      DU.key1.key1_min                    0.477
DU.key1.key2_median                      UU.key1.key2_median                 0.899
unique_key1_count                        unique_key2_count                   0.977
unique_transition_count                  consecutive_key1_repetition_rate    0.682
consecutive_key1_repetition_rate         backspace_rate                      0.809
backspace_rate                           consecutive_key1_repetition_rate    0.809


In [68]:
reduced_features = [
    "DU.key1.key1_median",
    "DU.key1.key2_median",
    "unique_key1_count",
    "unique_transition_count",
    "consecutive_key1_repetition_rate",
    "backspace_rate"
]

print("Reduced features:")
for feature in reduced_features:
    print(feature)

Reduced features:
DU.key1.key1_median
DU.key1.key2_median
unique_key1_count
unique_transition_count
consecutive_key1_repetition_rate
backspace_rate


In [69]:
reduced_corr = X[reduced_features].corr().abs()

upper_reduced = reduced_corr.where(
    np.triu(np.ones(reduced_corr.shape), k=1).astype(bool)
)

print("Correlation among reduced features:")
print("-----------------------------------")
print(upper_reduced.stack().sort_values(ascending=False))

Correlation among reduced features:
-----------------------------------
consecutive_key1_repetition_rate  backspace_rate                      0.808587
unique_transition_count           consecutive_key1_repetition_rate    0.681998
unique_key1_count                 unique_transition_count             0.670579
unique_transition_count           backspace_rate                      0.541666
unique_key1_count                 consecutive_key1_repetition_rate    0.524945
                                  backspace_rate                      0.459386
DU.key1.key1_median               DU.key1.key2_median                 0.406701
DU.key1.key2_median               consecutive_key1_repetition_rate    0.304033
DU.key1.key1_median               consecutive_key1_repetition_rate    0.272926
DU.key1.key2_median               unique_transition_count             0.251921
                                  backspace_rate                      0.233649
DU.key1.key1_median               unique_transition_count  

In [70]:
from sklearn.preprocessing import StandardScaler

# Reduced feature dataset
X_reduced = scored_keyboard_features[reduced_features]

# Use the same train/test indices as before
X_reduced_train = X_reduced.iloc[train_idx].copy()
X_reduced_test = X_reduced.iloc[test_idx].copy()

# Scale using training data only
reduced_scaler = StandardScaler()

X_reduced_train_scaled = reduced_scaler.fit_transform(X_reduced_train)
X_reduced_test_scaled = reduced_scaler.transform(X_reduced_test)

print("Reduced training shape:", X_reduced_train_scaled.shape)
print("Reduced testing shape:", X_reduced_test_scaled.shape)

Reduced training shape: (7971, 6)
Reduced testing shape: (2029, 6)


In [71]:
from sklearn.linear_model import LinearRegression

reduced_linear_model = LinearRegression()

reduced_linear_model.fit(
    X_reduced_train_scaled,
    y_train
)

print("Reduced Linear Regression model trained successfully.")

Reduced Linear Regression model trained successfully.


In [72]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Predictions
y_pred_reduced = reduced_linear_model.predict(
    X_reduced_test_scaled
)

# Metrics
mae_reduced = mean_absolute_error(y_test, y_pred_reduced)
mse_reduced = mean_squared_error(y_test, y_pred_reduced)
rmse_reduced = np.sqrt(mse_reduced)
r2_reduced = r2_score(y_test, y_pred_reduced)

print("Reduced Linear Regression Results")
print("----------------------------------")
print("MAE :", mae_reduced)
print("MSE :", mse_reduced)
print("RMSE:", rmse_reduced)
print("R²  :", r2_reduced)

Reduced Linear Regression Results
----------------------------------
MAE : 5.08278210373012
MSE : 42.96739381544553
RMSE: 6.554951854548249
R²  : 0.8060034627457799


In [73]:
reduced_coefficients = pd.DataFrame({
    "feature": reduced_features,
    "coefficient": reduced_linear_model.coef_
})

reduced_coefficients["absolute_coefficient"] = (
    reduced_coefficients["coefficient"].abs()
)

reduced_coefficients = reduced_coefficients.sort_values(
    "absolute_coefficient",
    ascending=False
)

print("Reduced Linear Regression Coefficients")
print("----------------------------------------")
print(reduced_coefficients)

Reduced Linear Regression Coefficients
----------------------------------------
                            feature  coefficient  absolute_coefficient
5                    backspace_rate    -6.001498              6.001498
0               DU.key1.key1_median    -5.995721              5.995721
2                 unique_key1_count     4.928534              4.928534
1               DU.key1.key2_median    -4.854645              4.854645
3           unique_transition_count     3.735964              3.735964
4  consecutive_key1_repetition_rate    -0.635630              0.635630


In [74]:
reduced_dataset = scored_keyboard_features[
    [
        "participant",
        "session",
        "window_id",
        *reduced_features,
        "productivity_score"
    ]
].copy()

print("Reduced dataset shape:", reduced_dataset.shape)
print("\nColumns:")
print(reduced_dataset.columns.tolist())

Reduced dataset shape: (10000, 10)

Columns:
['participant', 'session', 'window_id', 'DU.key1.key1_median', 'DU.key1.key2_median', 'unique_key1_count', 'unique_transition_count', 'consecutive_key1_repetition_rate', 'backspace_rate', 'productivity_score']


In [75]:
reduced_dataset.to_csv(
    "../data/processed/keyboard_reduced_productivity_dataset.csv",
    index=False
)

print("Reduced dataset saved successfully.")
print("Shape:", reduced_dataset.shape)

Reduced dataset saved successfully.
Shape: (10000, 10)


In [76]:
reduced_check = pd.read_csv(
    "../data/processed/keyboard_reduced_productivity_dataset.csv"
)

print("Shape:", reduced_check.shape)
print("\nMissing values:")
print(reduced_check.isnull().sum().sum())

print("\nDuplicate rows:")
print(reduced_check.duplicated().sum())

print("\nProductivity score range:")
print("Min:", reduced_check["productivity_score"].min())
print("Max:", reduced_check["productivity_score"].max())

Shape: (10000, 10)

Missing values:
0

Duplicate rows:
0

Productivity score range:
Min: 3.690476190476194
Max: 97.43589743589745
